# Time Pattern Extraction using DBSCAN

This notebook demonstrates how daily routines can be discovered using
Density-Based Spatial Clustering (DBSCAN).

Unlike the bucket-based approach, DBSCAN automatically discovers clusters
of similar event times and identifies noise events.

Special care is taken to handle circular time (midnight wraparound).

Example patterns:

- living_room_light usually turns ON around 00:00
- living_room_light usually turns ON around 04:02
- living_room_light usually turns ON around 18:02

## Step 1

Import required libraries.

We use:

- numpy for numerical operations
- sklearn DBSCAN for clustering
- statistics for pattern scoring
- datetime for timestamp processing

In [1]:
# imports
import math
import statistics

import numpy as np
from sklearn.cluster import DBSCAN

from collections import defaultdict
from datetime import datetime

In [3]:
# sample dataset
events = [

    # Midnight routine

    ("living_room_light", "ON", "2026-06-01 23:58"),
    ("living_room_light", "ON", "2026-06-02 00:01"),
    ("living_room_light", "ON", "2026-06-03 23:59"),
    ("living_room_light", "ON", "2026-06-04 00:02"),
    ("living_room_light", "ON", "2026-06-05 23:57"),
    ("living_room_light", "ON", "2026-06-06 00:03"),
    ("living_room_light", "ON", "2026-06-07 23:58"),
    ("living_room_light", "ON", "2026-06-08 00:01"),
    ("living_room_light", "ON", "2026-06-09 23:59"),

    # Morning routine

    ("living_room_light", "ON", "2026-06-01 04:01"),
    ("living_room_light", "ON", "2026-06-02 04:03"),
    ("living_room_light", "ON", "2026-06-03 04:00"),
    ("living_room_light", "ON", "2026-06-04 04:05"),
    ("living_room_light", "ON", "2026-06-05 04:02"),
    ("living_room_light", "ON", "2026-06-06 04:01"),
    ("living_room_light", "ON", "2026-06-07 04:04"),

    # Evening routine

    ("living_room_light", "ON", "2026-06-01 18:00"),
    ("living_room_light", "ON", "2026-06-02 18:02"),
    ("living_room_light", "ON", "2026-06-03 18:01"),
    ("living_room_light", "ON", "2026-06-04 18:03"),
    ("living_room_light", "ON", "2026-06-05 18:04"),
    ("living_room_light", "ON", "2026-06-06 18:02"),
    ("living_room_light", "ON", "2026-06-07 18:01"),
    ("living_room_light", "ON", "2026-06-08 18:03"),
    ("living_room_light", "ON", "2026-06-09 18:02"),
    ("living_room_light", "ON", "2026-06-10 18:00"),

    # Noise

    ("living_room_light", "ON", "2026-06-02 11:15"),
    ("living_room_light", "ON", "2026-06-11 22:30"),
]

In [5]:
# parse events

parsed_events = []

for device, action, ts in events:

    parsed_events.append(
        {
            "device": device,
            "action": action,
            "timestamp": datetime.strptime(
                ts,
                "%Y-%m-%d %H:%M"
            )
        }
    )
    
parsed_events

[{'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 1, 23, 58)},
 {'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 2, 0, 1)},
 {'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 3, 23, 59)},
 {'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 4, 0, 2)},
 {'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 5, 23, 57)},
 {'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 6, 0, 3)},
 {'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 7, 23, 58)},
 {'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 8, 0, 1)},
 {'device': 'living_room_light',
  'action': 'ON',
  'timestamp': datetime.datetime(2026, 6, 9, 23, 59)},
 {'device': 'living_room_light',
  'action': 'ON',
  '

## Step 2

Convert timestamps into minutes since midnight.

Examples:

00:01 -> 1

04:02 -> 242

18:02 -> 1082

23:58 -> 1438

In [6]:
# helper functions
def minutes_of_day(ts):

    return ts.hour * 60 + ts.minute


def fmt_hhmm(minutes):

    m = int(round(minutes)) % 1440

    return f"{m//60:02d}:{m%60:02d}"

## Why Circular Statistics?

Time-of-day is not linear.

23:59 and 00:01 are only 2 minutes apart.

Normal statistics treat them as 1438 minutes apart.

Circular statistics correctly handle midnight wraparound.

In [7]:
def circular_mean(minutes):

    angles = [
        2 * math.pi * m / 1440
        for m in minutes
    ]

    x = sum(math.cos(a) for a in angles)
    y = sum(math.sin(a) for a in angles)

    mean_angle = math.atan2(y, x)

    if mean_angle < 0:
        mean_angle += 2 * math.pi

    return (
        mean_angle
        * 1440
        / (2 * math.pi)
    )


def circular_distance(a, b):

    diff = abs(a - b)

    return min(
        diff,
        1440 - diff
    )


def circular_stddev(minutes):

    center = circular_mean(minutes)

    distances = [
        circular_distance(
            m,
            center
        )
        for m in minutes
    ]

    return statistics.pstdev(distances)

## Step 3

DBSCAN is applied independently to every:

(device, action)

combination.

In [8]:
groups = defaultdict(list)

for e in parsed_events:

    key = (
        e["device"],
        e["action"]
    )

    groups[key].append(
        minutes_of_day(
            e["timestamp"]
        )
    )

## Step 4

DBSCAN cannot directly understand circular time.

Instead every time value is projected onto a unit circle.

00:00 and 23:59 become neighboring points.

In [9]:
def minute_to_point(minute):

    angle = (
        2
        * math.pi
        * minute
        / 1440
    )

    return [
        math.cos(angle),
        math.sin(angle)
    ]

In [10]:
# build feature matarix

minutes = groups[
    ("living_room_light", "ON")
]

X = np.array(
    [
        minute_to_point(m)
        for m in minutes
    ]
)

X.shape

(28, 2)

## Step 5

DBSCAN parameters:

eps

Maximum neighborhood radius.

min_samples

Minimum observations required to form a cluster.

Noise points receive label:

-1

In [11]:
dbscan = DBSCAN(
    eps=0.03,
    min_samples=5
)

labels = dbscan.fit_predict(X)

labels

array([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  1,  1,  2,
        2,  2,  2,  2,  2,  2,  2,  2,  2, -1, -1])

In [12]:
# cluster inspection

for minute, label in zip(minutes, labels):

    print(
        fmt_hhmm(minute),
        "->",
        label
    )

23:58 -> 0
00:01 -> 0
23:59 -> 0
00:02 -> 0
23:57 -> 0
00:03 -> 0
23:58 -> 0
00:01 -> 0
23:59 -> 0
04:01 -> 1
04:03 -> 1
04:00 -> 1
04:05 -> 1
04:02 -> 1
04:01 -> 1
04:04 -> 1
18:00 -> 2
18:02 -> 2
18:01 -> 2
18:03 -> 2
18:04 -> 2
18:02 -> 2
18:01 -> 2
18:03 -> 2
18:02 -> 2
18:00 -> 2
11:15 -> -1
22:30 -> -1


## Step 6

DBSCAN automatically discovers:

Cluster 0 -> Midnight routine

Cluster 1 -> Morning routine

Cluster 2 -> Evening routine

Noise points are discarded.

In [13]:
clusters = {}

for minute, label in zip(
    minutes,
    labels
):

    if label == -1:
        continue

    clusters.setdefault(
        label,
        []
    ).append(minute)

clusters

{np.int64(0): [1438, 1, 1439, 2, 1437, 3, 1438, 1, 1439],
 np.int64(1): [241, 243, 240, 245, 242, 241, 244],
 np.int64(2): [1080, 1082, 1081, 1083, 1084, 1082, 1081, 1083, 1082, 1080]}

In [14]:
# scoring helpers

def support_score(count):

    return min(
        count / 10,
        1.0
    )


def consistency_score(
    stddev,
    tolerance=30
):

    return max(
        0,
        1 - stddev / tolerance
    )

In [16]:
# generate patterns

patterns = []

for cluster_id, cluster in clusters.items():

    mean_time = circular_mean(cluster)

    stddev = circular_stddev(cluster)

    support = support_score(
        len(cluster)
    )

    consistency = consistency_score(
        stddev
    )

    confidence = (
        support +
        consistency
    ) / 2

    patterns.append(
        {
            "cluster_id":
            cluster_id,

            "usual_time":
            fmt_hhmm(mean_time),

            "occurrences":
            len(cluster),

            "confidence":
            round(confidence,3)
        }
    )

patterns

[{'cluster_id': np.int64(0),
  'usual_time': '00:00',
  'occurrences': 9,
  'confidence': 0.937},
 {'cluster_id': np.int64(1),
  'usual_time': '04:02',
  'occurrences': 7,
  'confidence': 0.837},
 {'cluster_id': np.int64(2),
  'usual_time': '18:02',
  'occurrences': 10,
  'confidence': 0.988}]

# Conclusions

DBSCAN successfully discovered:

- Midnight routine
- Morning routine
- Evening routine

while automatically rejecting noise.

Advantages:

✓ No bucket creation

✓ No bucket merging

✓ Natural noise handling

✓ Automatic cluster discovery

✓ Midnight-safe using circular coordinates

Disadvantages:

✗ Requires parameter tuning

✗ Harder to explain than histogram clustering

✗ Less deterministic

The bucket-based and DBSCAN approaches achieve similar results on this dataset, but DBSCAN removes much of the manual clustering logic.

`We initially explored density-based clustering (DBSCAN) for routine discovery. However, smart-home routines are inherently time-based and highly interpretable. We therefore designed a custom multi-cluster temporal extraction engine that discovers multiple daily routines, handles midnight wraparound using circular statistics, filters noise, and assigns confidence scores to each learned behavior.`